# Deep Learning Midterm Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

## Training Model Version

Author: Thomas Kong

NetId: tk2558

Goal: Train Model

Important: This notebook only consists ofthe trianing model portion of the Notebook

## Referenced Data and Docs

### Dataset resources
- Provided train.csv

### Qwen 2B fine-tuning references
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

### **Section 0: Installing Necessary Packages**

> Make sure all packages and their versions are available and correct

> Uncomment whatever is needed to install for notebook environment

> Make sure train.csv and test.csv is uploaded for notebook to access

In [ ]:
# Uncomment the following in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml ftfy svgpathtools

# Install Node.js (if not already available)
# !apt-get update -y
# !apt-get install -y nodejs npm

# Install SVGO globally
!npm install -g svgo

In [ ]:
import unsloth, transformers, trl
# CHECK AVAILABLE AND CORRECT VERSIONS
print(transformers.__version__)
print(trl.__version__)
print(unsloth.__version__)

In [ ]:
# Install Node.js (if not already available)
!node -v # Verify Node
!svgo --version # Verify SVGO installation

### **Section 1: Configuration**

> Initialize variables for the notebook and models

> After running all cellblocks in Section 1, you can skip to 2B if you are already using pre-installed training_compressed.csv or skip to Section 7 if you are using pretrained fine-tuned model provided.

In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import hashlib, random, numpy as np, torch

NETID = "tk2558"
SEED  = int(hashlib.sha256(NETID.encode()).hexdigest(), 16) % 10000
print(f"NetID: {NETID}  |  Seed: {SEED}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
# "unsloth/Qwen3.5-2B-Instruct-bnb-4bit", (Does Not Exist)
# "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit" (VL Model, Attempted)
# "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" (Attempted)
# unsloth/Qwen3-4B-Base-unsloth-bnb-4bit (Attempted)
# "Qwen/Qwen2.5-Coder-3B-Instruct" (Attempted)
# "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit" (Current)

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 64,
    "learning_rate": 2e-4, #2e-4,
    "num_train_epochs": 1, #1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "warmup_steps": 200, #10,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 30000, # Adjust as necessary for GPU 
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
    #"output_dir": "/content/working/qwen2b_svg_lora",
}

CONFIG

In [ ]:
import pandas as pd

# LOAD DATASET
df = pd.read_csv("/kaggle/input/datasets/tk2558/train-prompt/train.csv", engine="python", on_bad_lines='skip')
assert all(col in df.columns for col in ["id", "prompt", "svg"])

print("Total samples:", len(df))
df = df.dropna(subset=["prompt", "svg"])
df.head()

### **Section 2B: Preprocessing Data Part 2**

> (You can either upload train_compressed.csv as an input for the notebook to access or use Public Available Version of train_compressed.csv that I created and uploaded to Hugging Face)

> Now that we have train_compressed.csv, we need to clean the data some more. If you look deeper at train_compression.csv you can noticed there are still some bad data (incorrect SVG outputs, encoding garbage in prompts, etc). This is the Data Cleaning Pipeline which helps to provide model better training data

In [ ]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    # "kaggle/train_compressed.csv": {
    #     "type": "csv",
    #     "path": "/kaggle/working/train_compressed.csv",
    #     #"path": "/kaggle/input/datasets/tk2558/train-compressed/train_compressed.csv", # ADDED AS INPUT
    #     "prompt_fields": ["prompt"],
    #     "svg_fields": ["svg"],
    # },
    
    "tk2558/train_compressed": {
        "split": "train",
        "prompt_fields": ["prompt"],
        "svg_fields": ["svg"],
    },
}

ACTIVE_SOURCES = [
    #"kaggle/train_compressed.csv", # USE LOCAL SAVED
    "tk2558/train_compressed", # USE PUBLIC HUGGING FACE VERSION
]

print("Dataset(s) Ready")

In [ ]:
from unsloth import FastLanguageModel
# Load Model (4bit for efficiency)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

print("\n=== CHECK TOKENS ===")
print("EOS token:", tokenizer.eos_token)       # should be <|im_end|>
print("EOS token ID:", tokenizer.eos_token_id) # should be 151645
print("PAD token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)
print("==================\n")

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""

def clean_svg(svg):
    svg = svg.strip()

    # Remove scripts / bad stuff
    svg = re.sub(r"<script.*?>.*?</script>", "", svg, flags=re.DOTALL)

    # Remove invalid tags (optional but helpful)
    svg = re.sub(r"</?(foreignObject|iframe|object).*?>", "", svg)

    # Ensure proper root
    if not svg.lower().startswith("<svg"):
        #print("SVG_start_missing")
        return ""

    # Ensure closing tag
    if not svg.endswith("</svg>"):
        #print("SVG_end_missing")
        return ""

    if len(svg) > 7950: # ONLY LESS THAN 8000 CHARACTERS
        #print("SVG_too_big")
        return ""

    if svg.count("<path") > 255: # ONLY LESS THAN 256
        #print("SVG_too_many_paths")
        return ""

    tokens = tokenizer.encode(svg)
    token_count = len(tokens)
    if token_count > 1024: # LESS THAN CONFIG['max_seq_length']
        #print("Token Too Big")
        return ""

    # FORCE CANVAS STANDARDIZATION (Only affects the root <svg> tag)
    svg = re.sub(r'(<svg[^>]*?\s)width="[^"]*"', r'\1width="256"', svg, count=1)
    svg = re.sub(r'(<svg[^>]*?\s)height="[^"]*"', r'\1height="256"', svg, count=1)
    svg = re.sub(r'(<svg[^>]*?\s)viewBox="[^"]*"', r'\1viewBox="0 0 256 256"', svg, count=1)

    return svg

def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)

    svg = clean_svg(svg)

    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}

def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")

    # CASE 1: HuggingFace dataset (if using external Datasets)
    if cfg.get("type", "hf") == "hf":
        if "data_files" in cfg:
            ds = load_dataset(
                dataset_id,
                data_files=cfg["data_files"],
                split=cfg["split"]
            )
        else:
            ds = load_dataset(dataset_id, split=cfg["split"])

    # CASE 2: Kaggle CSV
    elif cfg["type"] == "csv":
        #df = pd.read_csv(cfg["path"], engine="python", on_bad_lines="skip")
        df = pd.read_csv(cfg["path"])
        ds = Dataset.from_pandas(df)

    else:
        raise ValueError(f"Unknown dataset type for {dataset_id}")

    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

print("CLEANING FUNCTIONS READY")

### **Section 3 (IMPORTANT): Secret Key for Hugging Face API**

> Make sure notebook can access the secret key/token and you can assign it to an environment variable.

> If using Kaggle, uncomment top half of codeblock and comment the bottom. If using Google Colab, uncomment bottom half of codeblock and comment the top.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# --------------------------------------------------------------------- #

# from google.colab import userdata
# import os
# # Access the secret and assign it to an environment variable
# # Replace 'MY_API_KEY' with the exact name you used in the Secrets Manager
# api_key = userdata.get("HF_TOKEN")
# os.environ["HF_TOKEN"] = api_key

print("KEY READY")

### **Section 4: Load Dataset**

> Normalize Dataset and format it to SFT training style so it's ready for the model to learn from. Double check everything is functional!

In [ ]:
datasets_ok = []
datasets_origin = []

for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
        datasets_origin.append((source, ds))
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)

train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

In [ ]:
SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "When given a description, output ONLY a single valid SVG with these rules:\n"
    "1. Fill the full viewBox — shapes should be large and centered, not small or tucked into a corner.\n"
    "2. Use solid fills and simple strokes. No masks, filters, or external references.\n"
    "3. End output with </svg>.\n"
)

def format_sft_text(example):
    if not example["prompt"] or not example["svg"]:
        return {"text": ""}
    svg = example["svg"].strip().replace("\n", "")

    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}<|im_end|>\n"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
#train_text = train_ds.map(format_plain, remove_columns=train_ds.column_names)
train_text = train_text.filter(lambda x: x["text"] != "")

eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)
eval_text = eval_text.filter(lambda x: x["text"] != "")

In [ ]:
# CHECK FIRST VALUE IN DATASET
print(train_text[0]["text"])
print(len(train_text[0]["text"]))
print(len(tokenizer.encode(train_text[0]["text"])))

In [ ]:
# CHECK RANDOM VALUE IN DATASET
train_text_length = len(train_text)
ran_idx = random.randint(1, train_text_length - 1)

print(train_text[ran_idx]["text"])
print(len(train_text[ran_idx]["text"]))
print(len(tokenizer.encode(train_text[ran_idx]["text"])))

In [ ]:
def analyze_dataset_metrics(dataset, name, tokenizer):
    print(f"--- Metrics for {name} ---")

    char_counts = []
    token_counts = []
    path_counts = []

    for example in dataset:
        full_text = example["text"]
        char_counts.append(len(full_text))
        token_counts.append(len(tokenizer.encode(full_text)))

        # Extract SVG from the full formatted text and count paths
        svg_start_idx = full_text.rfind("<svg")
        svg_end_idx = full_text.rfind("</svg>")

        if svg_start_idx != -1 and svg_end_idx != -1 and svg_end_idx > svg_start_idx:
            svg_content = full_text[svg_start_idx : svg_end_idx + len("</svg>")]
            if (svg_content.count("<path")):
              path_counts.append(svg_content.count("<path"))

    if char_counts:
        print(f"Characters (full formatted text):")
        print(f"  Max: {max(char_counts):.0f}, Avg: {sum(char_counts) / len(char_counts):.2f}")
    if token_counts:
        print(f"Tokens (full formatted text):")
        print(f"  Max: {max(token_counts):.0f}, Avg: {sum(token_counts) / len(token_counts):.2f}")
    if path_counts:
        print(f"Path Counts (extracted SVG):")
        print(f"  Max: {max(path_counts):.0f}, Avg: {sum(path_counts) / len(path_counts):.2f}, Num: {len(path_counts)}")
    print("-" * (len(name) + 12))

analyze_dataset_metrics(train_text, "train_text", tokenizer)
analyze_dataset_metrics(eval_text, "eval_text", tokenizer)

### **Section 5: Training the Model**

> Run the training for the model

> (Running ignore warnings is **optional** if any Future Warnings appear that are cluttering the output cell block)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
print("Ignoring Future Warnings")

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    average_tokens_across_devices=False,
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    #warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    #gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
    assistant_only_loss=True, # MASKING
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

train_result = trainer.train()
train_result

### **Section 6: Saving the Model**

> Save the pre-trained model for future use

> You can local save, save to google drive or saved to HuggingFace with a **WRITE** API Key

> (Uncomment respective codeblock for whichever you want to save to. You can also save to all three)


In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"]) # LOCAL SAVING
tokenizer.save_pretrained(CONFIG["output_dir"]) # LOCAL SAVING

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
# CLOUD SAVING
# from google.colab import drive
# drive.mount('/content/drive')

# model.save_pretrained("/content/drive/MyDrive/svg_model_coder") # SAVE FILE TO DRIVE
# tokenizer.save_pretrained("/content/drive/MyDrive/svg_model_coder") # SAVE FILE TO DRIVE

In [ ]:
# CLOUD SAVING TO HF
# api_key = userdata.get("HF_TOKEN_WRITE") # USE HF TOKEN KEY WITH WRITING PERMISSIONS
# os.environ["HF_TOKEN_WRITE"] = api_key # USE HF TOKEN KEY WITH WRITING PERMISSIONS

# model.push_to_hub("tk2558/qwen_lora_midterm", token = api_key) # Online saving to HF
# tokenizer.push_to_hub("tk2558/qwen_lora_midterm", token = api_key) # Online saving to HF